# Session 5. Long-term memory

**The assistant that knows you after the process dies.**

- checkpointer = one conversation; store = the person across conversations
- two writers: an extractor after the turn, or the agent itself with tools

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## What session 4 left behind

**SqliteSaver remembers threads. A new thread_id is a stranger.**

- checkpointer state lives under one thread_id: a conversation, resumable
- nothing carries the guest from yesterday's visit into today's
- today: a second memory, keyed by the person, not the conversation

In [ ]:
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph


def count_history(state: MessagesState) -> dict:
    # a fake barista: reports how much history this thread carries
    return {"messages": [AIMessage(f"messages in this thread: {len(state['messages'])}")]}


builder = StateGraph(MessagesState)
builder.add_node("count", count_history)
builder.add_edge(START, "count")
builder.add_edge("count", END)
counter = builder.compile(checkpointer=InMemorySaver())

for thread in ("visit-a", "visit-a", "visit-b"):  # same thread twice, then a new one
    out = counter.invoke(
        {"messages": [("user", "hi")]},
        config={"configurable": {"thread_id": thread}, "recursion_limit": 5},
    )
    print(thread, "->", out["messages"][-1].content)

## Three kinds of memory

**Semantic, episodic, procedural — in coffee terms.**

- semantic: facts about the guest — name, usual drink, the allergy
- episodic: what happened — last visit she returned a burnt espresso
- procedural: standing instructions — always offer the loyalty card
- module-5 and Agent_Memory_Techniques 06-11 use exactly these terms

## The Store

**Namespaces are tuples, like folders. Keys are file names. Values are dicts.**

- put and get address one item; search lists a namespace prefix
- no model anywhere in this API: it is a small document database

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
GUEST = ("profiles", "guest-7")  # a tuple, one segment per folder level

store.put(GUEST, "profile", {"name": "Dasha", "usual_drink": "flat white"})
store.put(("visits", "guest-7", "2026-08"), "v1", {"note": "asked for oat milk"})

item = store.get(GUEST, "profile")
print("value:  ", item.value)  # the dict you put, exactly
print("key:    ", item.key)
print("created:", item.created_at)

print([hit.key for hit in store.search(("visits", "guest-7"))])  # prefix, any depth
print([hit.key for hit in store.search(("profiles",), filter={"name": "Dasha"})])

**`query=` without an index looks like search. It is a listing.**

- no error, no warning: every item comes back, score None
- the day you add semantic search to your assistant, this is the bug

In [ ]:
memories = InMemoryStore()  # no index configured, on purpose
memories.put(("memories", "guest-7"), "m1", {"fact": "dairy allergy, oat milk only"})
memories.put(("memories", "guest-7"), "m2", {"fact": "prefers the window seat"})
memories.put(("memories", "guest-7"), "m3", {"fact": "pays by card, no receipt"})

for hit in memories.search(("memories", "guest-7"), query="what milk does she take"):
    print(hit.score, "|", hit.value["fact"])  # every fact, unranked, nothing raised

In [ ]:
def bucket_embed(texts: list[str]) -> list[list[float]]:
    """Hash words into 16 buckets. A toy stand-in for an embedding model."""
    vectors = []
    for text in texts:
        vec = [0.0] * 16
        for word in text.lower().split():
            vec[sum(word.encode()) % 16] += 1.0  # stable across runs, unlike hash()
        vectors.append(vec)
    return vectors


memories = InMemoryStore(index={"embed": bucket_embed, "dims": 16})
memories.put(("memories", "guest-7"), "m1", {"fact": "dairy allergy, oat milk only"})
memories.put(("memories", "guest-7"), "m2", {"fact": "prefers the window seat"})
memories.put(("memories", "guest-7"), "m3", {"fact": "pays by card, no receipt"})

for hit in memories.search(("memories", "guest-7"), query="what milk does she take"):
    print(round(hit.score, 3), "|", hit.value["fact"])

**Ranked. Sixteen buckets is a toy; the wiring is not.**

- the course env has no embedding endpoint, so the demo hashes words
- production passes a real embedding model in the same `index=` slot
- collisions still score: the card fact matched by accident; real vectors separate better

## A store from scratch

**Technique 21: the whole API is a dict and a filter loop.**

- `(namespace, key)` as the dict key — that is the entire trick
- fifteen lines below reproduce put, get and prefix search

In [ ]:
from types import SimpleNamespace


class MiniStore:
    def __init__(self):
        self.items = {}  # (namespace, key) -> value; the whole storage

    def put(self, namespace, key, value):
        self.items[(namespace, key)] = value

    def get(self, namespace, key):
        value = self.items.get((namespace, key))
        return SimpleNamespace(key=key, value=value) if value else None

    def search(self, prefix, filter=None):
        return [
            SimpleNamespace(key=key, value=value, score=None)
            for (namespace, key), value in self.items.items()
            if namespace[: len(prefix)] == prefix
            and all(value.get(f) == v for f, v in (filter or {}).items())
        ]


mini = MiniStore()
mini.put(GUEST, "profile", {"name": "Dasha", "usual_drink": "flat white"})
mini.put(("visits", "guest-7", "2026-08"), "v1", {"note": "asked for oat milk"})
print(mini.get(GUEST, "profile").value)
print([hit.key for hit in mini.search(("visits",))])

**InMemoryStore is that dict, inside this process.**

- kernel restart, redeploy, crash: the guest is gone
- long-term memory that dies with the process is short-term memory

In [ ]:
fresh = InMemoryStore()  # what a restarted process would build
print(fresh.get(GUEST, "profile"))  # None: the profile lived in the old object

**Session 4's move again, one shelf up.**

- threads went InMemorySaver -> SqliteSaver; guests go InMemoryStore -> SqliteStore
- same package, `langgraph-checkpoint-sqlite`, already installed
- PostgresStore is the deploy-scale shelf — the same InMemory -> Sqlite -> Postgres move session 8 makes for the checkpointer with `AsyncPostgresSaver`
- importing PostgresStore needs libpq, not today

In [ ]:
import sqlite3
from pathlib import Path

from langgraph.store.sqlite import SqliteStore

DB = Path("coffee-memory.db")
DB.unlink(missing_ok=True)  # clean rerun every time; keep *.db out of git

# isolation_level=None or put raises "cannot start a transaction"
conn = sqlite3.connect(DB, isolation_level=None, check_same_thread=False)
STORE = SqliteStore(conn)
STORE.setup()  # creates the tables on first use

STORE.put(("shop",), "sticky-note", {"text": "we are out of oat milk"})
conn.close()  # the process "dies" here

conn = sqlite3.connect(DB, isolation_level=None, check_same_thread=False)
STORE = SqliteStore(conn)  # a restarted process starts exactly like this
print(STORE.get(("shop",), "sticky-note").value)

## Two memories in one graph

**`compile(checkpointer=..., store=...)`: the thread and the person, side by side.**

- `get_store()` reaches the store from inside nodes and inside tools
- docs also inject it: `Runtime` in nodes, `ToolRuntime` in tools — same store, another door
- the checkpointer keys on thread_id; the store namespace keys on the guest

In [ ]:
from langgraph.config import get_store

bot = chat_model("strong")  # the conversation runs on the strong model


def barista(state: MessagesState) -> dict:
    item = get_store().get(GUEST, "profile")  # the graph hands the store over
    known = item.value if item else "first visit, nothing known yet"
    prompt = f"You are the barista. Guest card: {known}. Serve warmly, briefly."
    reply = bot.invoke([{"role": "system", "content": prompt}] + state["messages"])
    return {"messages": [reply]}  # the reply object itself, never a rebuild


builder = StateGraph(MessagesState)
builder.add_node("barista", barista)
builder.add_edge(START, "barista")
builder.add_edge("barista", END)
coffee = builder.compile(checkpointer=InMemorySaver(), store=STORE)

In [ ]:
visit1 = coffee.invoke(
    {"messages": [("user", "Hi! A flat white with oat milk, please. I'm Dasha.")]},
    config={"configurable": {"thread_id": "visit-1"}, "recursion_limit": 6},
)
print(visit1["messages"][-1].content)

print("profile now:", STORE.get(GUEST, "profile"))  # still None. Who writes it?

**The bot read the profile. Nobody wrote it. Two ways to fix that.**

- background: an extractor distills the turn after the reply went out
- hot path: the agent itself saves through tools, mid-conversation
- docs call this choice the update strategy; today both, in this order

## Trustcall: the background writer

**Why not "just ask for JSON": regeneration loses fields.**

- re-emitting a long profile drops what the turn never mentioned
- trustcall patches the existing doc and validates, with retries
- extraction fills a form, so it runs on the cheap model

In [ ]:
from pydantic import BaseModel
from trustcall import create_extractor


class Profile(BaseModel):
    """What the coffee shop remembers about one guest."""

    name: str | None = None
    usual_drink: str | None = None
    milk: str | None = None


extractor = create_extractor(chat_model("cheap"), tools=[Profile], tool_choice="Profile")

result = extractor.invoke(
    {"messages": visit1["messages"]},  # the whole turn is the source
    config={"recursion_limit": 6},
)
print(result["responses"][0])
print(result["response_metadata"])

In [ ]:
profile = result["responses"][0].model_dump()
STORE.put(GUEST, "profile", profile)  # the notebook writes; trustcall only extracts

print(STORE.get(GUEST, "profile").value)

**The patch move: hand over what you already know.**

- `existing=` gives trustcall the current doc
- the model emits JSON Patch operations against it, not a fresh doc
- untouched fields survive; `json_doc_id` in the metadata names the patched doc

In [ ]:
update = extractor.invoke(
    {
        "messages": [("user", "Actually, make my usual a cortado from now on.")],
        "existing": {"Profile": profile},  # the doc the patches apply to
    },
    config={"recursion_limit": 6},
)
profile = update["responses"][0].model_dump()
print(profile)  # name and milk survived a turn that never mentioned them
print(update["response_metadata"])

STORE.put(GUEST, "profile", profile)

In [ ]:
visit2 = coffee.invoke(
    {"messages": [("user", "Morning! The usual, please.")]},
    config={"configurable": {"thread_id": "visit-2"}, "recursion_limit": 6},
)
print(visit2["messages"][-1].content)

print("history length:", len(visit2["messages"]))  # a fresh thread, yet she is known

**A profile flattens. Some memory is a list of facts.**

- "brings her own cup on Tuesdays" is not a profile field
- a collection: one small doc per fact; patch old ones, insert new ones
- `enable_inserts=True`, and `existing` becomes (id, tool, doc) tuples
- without a one-fact-per-memory rule the model merges everything into one doc

In [ ]:
import uuid


class Memory(BaseModel):
    """One remembered fact about the guest."""

    fact: str


collector = create_extractor(chat_model("cheap"), tools=[Memory], enable_inserts=True)

STORE.put(("memories", "guest-7"), "0", {"fact": "prefers the window seat"})
known = [  # what the store holds now, as (id, tool, doc) tuples
    (hit.key, "Memory", hit.value) for hit in STORE.search(("memories", "guest-7"))
]

RULES = "Update existing memories and add new ones. One fact per memory."  # or it merges
batch = collector.invoke(
    {
        "messages": [("system", RULES),
                     ("user", "Make that the corner table, not the window. Two more "
                              "things: after 16:00 I drink decaf, and on Tuesdays "
                              "I bring my own cup.")],
        "existing": known,
    },
    config={"recursion_limit": 6},
)
for doc, meta in zip(batch["responses"], batch["response_metadata"]):
    key = meta.get("json_doc_id") or uuid.uuid4().hex[:8]  # patched docs keep their id
    STORE.put(("memories", "guest-7"), key, doc.model_dump())
    print("patched" if "json_doc_id" in meta else "new    ", key, "->", doc.model_dump())

## Memory as tools

**Pattern two: the agent wields the memory itself.**

- technique 23, the memory-agent / task_mAIstro pattern
- save and search are ordinary session-2 tools: no new wire format, nothing new in the tool-calling machinery
- the model decides what is worth keeping, during the turn

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def save_memory(fact: str) -> str:
    """Store one durable fact about the guest for future visits."""
    key = uuid.uuid4().hex[:8]
    get_store().put(("memories", "guest-7"), key, {"fact": fact})  # same door as nodes
    return f"saved as {key}"


@tool
def search_memories(query: str) -> str:
    """Look up stored facts about the guest."""
    # STORE has no index: an honest listing; the model filters
    hits = get_store().search(("memories", "guest-7"), limit=5)
    if not hits:
        return "nothing stored about this guest yet"
    return "; ".join(hit.value["fact"] for hit in hits)


TOOLS = [save_memory, search_memories]
bound = chat_model("strong").bind_tools(TOOLS)


def call_model(state: MessagesState) -> dict:
    item = get_store().get(GUEST, "profile")
    known = item.value if item else "first visit"
    prompt = f"You are the barista. Guest card: {known}. Save facts worth keeping."
    reply = bound.invoke([{"role": "system", "content": prompt}] + state["messages"])
    return {"messages": [reply]}


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")
agent = builder.compile(checkpointer=InMemorySaver(), store=STORE)

print(agent.get_graph().draw_mermaid())

In [ ]:
visit3 = agent.invoke(
    {"messages": [("user", "Remember this: I have a dairy allergy, oat milk only.")]},
    config={"configurable": {"thread_id": "visit-3"}, "recursion_limit": 6},
)
for message in visit3["messages"]:  # the save_memory call, in the open
    message.pretty_print()

In [ ]:
visit4 = agent.invoke(
    {"messages": [("user", "Quick check: what milk do I take?")]},
    config={"configurable": {"thread_id": "visit-4"}, "recursion_limit": 6},
)
print(visit4["messages"][-1].content)  # a new thread, and the answer cites the fact

**The two writers, side by side.**

- background extraction never blocks a reply and never forgets to run
- tools cost a model turn per save, but the agent searches on demand
- hot path for what the user asked to keep; background for the rest
- task_mAIstro runs both, and patches its own instructions: procedural memory, closing the classification

## Observability

**One handler, one trace: watch the agent decide to save.**

- a fresh `CallbackHandler` per run gives one trace per run
- find the `save_memory` span inside `tools`: the hot path, on record
- `flush()`, as always: a notebook kernel never exits on its own

In [ ]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler

client = get_client()  # reads LANGFUSE_HOST and both keys from the environment
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", client.auth_check())

handler = CallbackHandler()
traced = agent.invoke(
    {"messages": [("user", "Remember: I switched to decaf on weekdays.")]},
    config={
        "configurable": {"thread_id": "visit-5"},
        "recursion_limit": 6,
        "callbacks": [handler],
    },
)
client.flush()  # nothing sends without this in a notebook

print(traced["messages"][-1].content)

## Memory frameworks

**When to stop hand-rolling: four libraries automate today's demo.**

- each automates extract, dedupe, retrieve — what we wrote by hand
- none installs today: each brings notable environment requirements
- condensed from Agent_Memory_Techniques docs/comparison.md (Apache-2.0): https://github.com/NirDiamant/Agent_Memory_Techniques/blob/main/docs/comparison.md

| Framework | Automates | Environment it needs | Technique |
|---|---|---|---|
| Mem0 | extraction and dedup as a drop-in memory layer | vector store, optional graph DB | 24 |
| Letta | an agent with self-editing memory blocks | its own server runtime | 25 |
| Zep | temporal knowledge graph over sessions | hosted service or own server | 26 |
| Graphiti | the temporal graph engine itself, DIY | Neo4j-class graph database | 27 |

## Practice

**Long-term memory in your own assistant, in your own repository.**

1. compile with both `checkpointer=` and `store=`; `create_agent` accepts both
2. a `Profile` schema for your user, trustcall-patched after each turn
3. `save_memory` / `search_memories` tools over the same store
4. namespace every put, get and search by user id

**Required artifact: a NEW dialogue recognizes you.**

- dialogue one teaches the agent; dialogue two, fresh thread_id, greets you informed
- commit `runs/session-05.md`: both dialogues, plus the trace export of the recognition run
- stretch: move the store to SqliteStore and survive a kernel restart
- keep `*.db` out of git

**Project: the memory section is mandatory in the architecture description.**

- name what your agent keeps: semantic, episodic, procedural, and where it lives
- from today, practice may land on the project agent; the assistant stays the fallback
- theme approval happens at the end of today's session

## Next time

**The agent learns to pause.**

- streaming token by token, and asking a human before acting